<a href="https://colab.research.google.com/github/yasmeentalata/prediction-achat-en-ligne/blob/main/prediction_achat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("online_shoppers_intention.csv")

print("Nombre de lignes et de colonnes :", df.shape)
display(df.head())
print("\nAchats et non-achats :")
print(df["Revenue"].value_counts())

Nombre de lignes et de colonnes : (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



Achats et non-achats :
Revenue
False    10422
True      1908
Name: count, dtype: int64


In [2]:
print("Taux d'achat :", round(df["Revenue"].mean() * 100, 1), "%")
print("Valeurs manquantes par colonne :")
print(df.isna().sum())
print("Types des colonnes :")
print(df.dtypes)

Taux d'achat : 15.5 %
Valeurs manquantes par colonne :
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64
Types des colonnes :
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
Oper

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

colonnes = ["Administrative", "Informational", "ProductRelated",
            "BounceRates", "ExitRates", "SpecialDay"]

X = df[colonnes]       # Informations utilisées pour prédire
y = df["Revenue"]      # Résultat à prédire : achat ou non

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
predictions = baseline.predict(X_test)

print("Sessions d'entraînement :", len(X_train))
print("Sessions de test :", len(X_test))
print("Accuracy :", round(accuracy_score(y_test, predictions), 3))
print("Rappel des achats :", round(recall_score(y_test, predictions), 3))

Sessions d'entraînement : 9864
Sessions de test : 2466
Accuracy : 0.845
Rappel des achats : 0.0


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

modele = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000)
)

modele.fit(X_train, y_train)
predictions_modele = modele.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, predictions_modele), 3))
print("Précision des achats :", round(precision_score(y_test, predictions_modele), 3))
print("Rappel des achats :", round(recall_score(y_test, predictions_modele), 3))
print("Matrice de confusion :")
print(confusion_matrix(y_test, predictions_modele))


Accuracy : 0.566
Précision des achats : 0.22
Rappel des achats : 0.707
Matrice de confusion :
[[1125  959]
 [ 112  270]]


donc finalement la pondération des classes augmente le rappel des achats, mais la précision reste faible. Il faut étudier ce compromis avant de choisir notre modèle.


In [5]:
modele_sans_ponderation = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

modele_sans_ponderation.fit(X_train, y_train)
predictions_sans_ponderation = modele_sans_ponderation.predict(X_test)

print("Précision des achats :", round(precision_score(y_test, predictions_sans_ponderation, zero_division=0), 3))
print("Rappel des achats :", round(recall_score(y_test, predictions_sans_ponderation), 3))
print(confusion_matrix(y_test, predictions_sans_ponderation))

Précision des achats : 0.4
Rappel des achats : 0.005
[[2081    3]
 [ 380    2]]


Sans pondération, le modèle annonce très rarement un achat et en manque 380 sur 382. Avec class_weight="balanced", il en détecte 270, mais produit beaucoup plus de fausses alertes. Le choix dépendra de l’importance accordée à ces deux types d’erreurs.

In [6]:
probabilites = modele.predict_proba(X_test)[:, 1]

resultats = X_test.head().copy()
resultats["Achat réel"] = y_test.head().values
resultats["Probabilité prédite"] = probabilites[:5].round(3)
resultats["Achat prédit"] = predictions_modele[:5]

display(resultats)


,Administrative,Informational,ProductRelated,BounceRates,ExitRates,SpecialDay,Achat réel,Probabilité prédite,Achat prédit
4722,1,0,13,0.024615,0.061538,0.6,False,0.119,False
6835,1,0,23,0.000000,0.033333,0.0,False,0.430,False
5524,0,0,6,0.028571,0.028571,0.0,True,0.468,False
663,0,0,2,0.000000,0.100000,0.0,False,0.067,False
136,0,0,9,0.005556,0.046296,0.0,False,0.316,False
